# Inspect and pin the base model

**Feature:** NB-001  
**Goal:** resolve one reproducible experiment configuration, inspect the supported model structure, and record immutable configuration and weight fingerprints.

The default smoke path creates a deterministic tiny Qwen3 model locally. It downloads no model or tokenizer data. Set `ANCHORED_MOE_SMOKE_TEST=0` only after supplying an exact model revision through the environment.

## Resolve configuration

All later cells consume this single resolved configuration. Production runs must replace the synthetic identifiers with exact Hugging Face repository revisions.

In [ ]:
import os

import torch
from transformers import AutoModelForCausalLM, Qwen3Config, Qwen3ForCausalLM

from anchored_moe import AnchoredMoEConfig
from anchored_moe.backends import Qwen3Backend
from anchored_moe.evaluation import fingerprint_module
from anchored_moe.utils import sha256_json

SMOKE_TEST = os.environ.get("ANCHORED_MOE_SMOKE_TEST", "1") != "0"
EXPERIMENT = {
    "seed": 7,
    "base_model_id": (
        "local-random-qwen3-smoke"
        if SMOKE_TEST
        else os.environ.get("ANCHORED_MOE_BASE_MODEL_ID", "Qwen/Qwen3-4B")
    ),
    "base_model_revision": (
        "synthetic-seed-7"
        if SMOKE_TEST
        else os.environ.get("ANCHORED_MOE_BASE_MODEL_REVISION")
    ),
    "converted_layers": "upper_half",
    "dtype": "float32",
}
if not SMOKE_TEST and not EXPERIMENT["base_model_revision"]:
    raise ValueError(
        "Set ANCHORED_MOE_BASE_MODEL_REVISION to an exact Hub revision."
    )

EXPERIMENT

## Load the anchor

Smoke mode uses a small random configuration suitable for CPU execution. The production path uses the same pinned repository and revision recorded above.

In [ ]:
torch.manual_seed(EXPERIMENT["seed"])
if SMOKE_TEST:
    base_config = Qwen3Config(
        vocab_size=64,
        hidden_size=16,
        intermediate_size=32,
        num_hidden_layers=4,
        num_attention_heads=4,
        num_key_value_heads=2,
        head_dim=4,
        max_position_embeddings=32,
    )
    base_model = Qwen3ForCausalLM(base_config).eval()
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        EXPERIMENT["base_model_id"],
        revision=EXPERIMENT["base_model_revision"],
        torch_dtype=torch.float32,
    ).eval()
    base_config = base_model.config

base_config_sha256 = sha256_json(base_config.to_dict())
base_weight_sha256 = fingerprint_module(base_model)
anchored_config = AnchoredMoEConfig(
    base_model_name_or_path=EXPERIMENT["base_model_id"],
    base_model_revision=EXPERIMENT["base_model_revision"],
    base_config_sha256=base_config_sha256,
    tokenizer_name_or_path=EXPERIMENT["base_model_id"],
    tokenizer_revision=EXPERIMENT["base_model_revision"],
    backend="qwen3",
    converted_layers=EXPERIMENT["converted_layers"],
)

## Inspect the backend contract

Discovery is read-only. It verifies the exact Qwen3 family contract and records the stable decoder-layer order without converting the model.

In [ ]:
backend = Qwen3Backend()
assert backend.supports_model(base_model)
mlp_references = tuple(backend.iter_mlp_references(base_model))
resolved_layers = anchored_config.resolve_converted_layers(len(mlp_references))
trainable_parameters = sum(
    parameter.numel() for parameter in base_model.parameters() if parameter.requires_grad
)
total_parameters = sum(parameter.numel() for parameter in base_model.parameters())

assert tuple(reference.layer_index for reference in mlp_references) == tuple(
    range(len(mlp_references))
)
assert len(base_config_sha256) == 64
assert len(base_weight_sha256) == 64
assert resolved_layers

inspection_summary = {
    "feature_id": "NB-001",
    "smoke_test": SMOKE_TEST,
    "model_type": base_config.model_type,
    "decoder_layers": len(mlp_references),
    "converted_layers": resolved_layers,
    "total_parameters": total_parameters,
    "trainable_parameters_before_conversion": trainable_parameters,
    "base_config_sha256": base_config_sha256,
    "base_weight_sha256": base_weight_sha256,
}
print("NB-001 validation passed", inspection_summary)